In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_squared_error, r2_score
from scipy import stats

DATA_PATH = Path(r"C:\Users\asus\Article-10-06-2026-ML-portfoliov1")

# Load saved ML predictions if available, otherwise load returns
try:
    metrics_unc = pd.read_csv(DATA_PATH / 'portfolio_metrics_unconstrained.csv', index_col=0)
    metrics_cap = pd.read_csv(DATA_PATH / 'portfolio_metrics_40cap.csv', index_col=0)
    compare_df  = pd.read_csv(DATA_PATH / 'portfolio_comparison_uncap_vs_cap.csv', index_col=[0,1])
    bl_mu       = pd.read_csv(DATA_PATH / 'bl_mu_vectors.csv')
    ml_mu       = pd.read_csv(DATA_PATH / 'ml_mu_vector.csv')

    print("=== PORTFOLIO METRICS — UNCONSTRAINED ===")
    print(metrics_unc.to_string())
    print("\n=== PORTFOLIO METRICS — 40% CAP ===")
    print(metrics_cap.to_string())
    print("\n=== DIRECT COMPARISON ===")
    print(compare_df.to_string())
    print("\n=== ML MU VECTOR ===")
    print(ml_mu.to_string())
    print("\n=== BL MU VECTORS ===")
    print(bl_mu.to_string())

except Exception as e:
    print(f"Error loading: {e}")
    import os
    print("\nFiles in ARTICLE DATA folder:")
    for f in os.listdir(DATA_PATH):
        print(f" -", f)

Error loading: [Errno 2] No such file or directory: 'C:\\Users\\asus\\Article-10-06-2026-ML-portfoliov1\\portfolio_metrics_unconstrained.csv'

Files in ARTICLE DATA folder:
 - .ipynb_checkpoints
 - 0000-Check-data-article_revised.ipynb
 - 000Final_ml-Articlev3_revised.ipynb
 - 000Final_portfolio_Articlev3_revised.ipynb
 - 000_Statistical_Validation.ipynb
 - view_dl_results_revised.ipynb


In [2]:
try:
    cum_unc = pd.read_csv(DATA_PATH / 'cumulative_returns_unconstrained.csv',
                          index_col=0, parse_dates=True)
    cum_cap = pd.read_csv(DATA_PATH / 'cumulative_returns_40cap.csv',
                          index_col=0, parse_dates=True)

    print("=== FINAL CUMULATIVE RETURNS — UNCONSTRAINED ===")
    print((cum_unc.iloc[-1] - 1).mul(100).round(2).rename('Total Return (%)'))

    print("\n=== FINAL CUMULATIVE RETURNS — 40% CAP ===")
    print((cum_cap.iloc[-1] - 1).mul(100).round(2).rename('Total Return (%)'))

    print(f"\nOut-of-sample period: {cum_unc.index.min().date()} → {cum_unc.index.max().date()}")
    print(f"Total OOS days: {len(cum_unc)}")

    print("\n=== COLUMN NAMES UNCONSTRAINED ===")
    print(cum_unc.columns.tolist())

    print("\n=== COLUMN NAMES 40% CAP ===")
    print(cum_cap.columns.tolist())

except Exception as e:
    print(f"Error: {e}")

Error: [Errno 2] No such file or directory: 'C:\\Users\\asus\\Article-10-06-2026-ML-portfoliov1\\cumulative_returns_unconstrained.csv'


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from scipy import stats

DATA_PATH = Path(r"C:\Users\asus\OneDrive\Рабочий стол\Diploma work\ARTICLE DATA\Preprocessed")

EQUITY_TICKERS = ['ASBN', 'CCBN', 'HSBK', 'KCEL', 'KEGC', 'KZTK', 'KZTO', 'KZAP']
FACTOR_TICKERS = ['F BRENT', 'F KASE', 'F USD_KZT']
ALL_TICKERS    = EQUITY_TICKERS + FACTOR_TICKERS
HORIZON        = 21
SEED           = 42

# ── Load data ─────────────────────────────────────────────────────────────────
raw = {}
for ticker in ALL_TICKERS:
    df = pd.read_csv(DATA_PATH / f"{ticker}.csv",
                     parse_dates=['Date'], index_col='Date')
    raw[ticker] = df[['Close']]

close_prices = pd.concat(
    [raw[t].rename(columns={'Close': t}) for t in ALL_TICKERS],
    axis=1, join='inner'
).sort_index().ffill().bfill()

log_returns = np.log(close_prices / close_prices.shift(1)).dropna()

# ── Simple feature matrix (momentum + vol + factors) ─────────────────────────
def build_features(ticker):
    ret = log_returns[ticker]
    f   = pd.DataFrame(index=log_returns.index)
    for w in [5, 10, 21]:
        f[f'ret_{w}d'] = ret.rolling(w).sum()
        f[f'mom_{w}d'] = close_prices[ticker] / close_prices[ticker].shift(w) - 1
    for w in [5, 21, 63]:
        f[f'vol_{w}d'] = ret.rolling(w).std() * np.sqrt(246)
    f['skew_21d'] = ret.rolling(21).skew()
    f['kurt_21d'] = ret.rolling(21).kurt()
    # RSI
    delta = close_prices[ticker].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    f['rsi_14'] = 100 - (100 / (1 + gain / (loss + 1e-9)))
    # MACD
    f['macd'] = (close_prices[ticker].ewm(span=12).mean() -
                 close_prices[ticker].ewm(span=26).mean())
    # Macro factors
    for ft in FACTOR_TICKERS:
        f[f'ret_{ft}'] = log_returns[ft]
    # Target
    f['target'] = ret.rolling(HORIZON).sum().shift(-HORIZON)
    f.dropna(inplace=True)
    X = f.drop(columns=['target'])
    y = f['target']
    return X, y

# ── Models ────────────────────────────────────────────────────────────────────
MODELS = {
    'ElasticNet': Pipeline([('sc', RobustScaler()),
                             ('m', ElasticNet(alpha=0.05, l1_ratio=0.83,
                                              max_iter=5000, random_state=SEED))]),
    'SVR':        Pipeline([('sc', RobustScaler()),
                             ('m', SVR(C=1.93, epsilon=0.03, gamma='auto'))]),
    'KNN':        Pipeline([('sc', RobustScaler()),
                             ('m', KNeighborsRegressor(n_neighbors=19))]),
    'RandomForest': RandomForestRegressor(n_estimators=485, max_depth=5,
                                          min_samples_leaf=7, random_state=SEED),
    'XGBoost':    XGBRegressor(n_estimators=287, max_depth=6,
                                learning_rate=0.06, random_state=SEED, verbosity=0),
    'LightGBM':   LGBMRegressor(n_estimators=251, max_depth=6,
                                  learning_rate=0.14, random_state=SEED, verbose=-1),
}

def da_test(y_true, y_pred):
    n       = len(y_true)
    correct = np.sum(np.sign(y_true) == np.sign(y_pred))
    p_hat   = correct / n
    z       = (p_hat - 0.5) / np.sqrt(0.25 / n)
    p_val   = 1 - stats.norm.cdf(z)
    return round(p_hat, 4), round(z, 3), round(p_val, 4), p_val < 0.05

# ── Walk-forward evaluation ───────────────────────────────────────────────────
tscv = TimeSeriesSplit(n_splits=4, gap=HORIZON)

results = []
for ticker in EQUITY_TICKERS:
    X, y = build_features(ticker)
    for model_name, model in MODELS.items():
        all_true, all_pred = [], []
        for tr_idx, te_idx in tscv.split(X):
            Xtr, Xte = X.iloc[tr_idx], X.iloc[te_idx]
            ytr, yte = y.iloc[tr_idx], y.iloc[te_idx]
            import copy
            m = copy.deepcopy(model)
            m.fit(Xtr, ytr)
            pred = m.predict(Xte)
            all_true.extend(yte.values)
            all_pred.extend(pred)
        yt = np.array(all_true)
        yp = np.array(all_pred)
        rmse = np.sqrt(mean_squared_error(yt, yp))
        r2   = r2_score(yt, yp)
        da, z, pval, sig = da_test(yt, yp)
        results.append({
            'Ticker': ticker, 'Model': model_name,
            'RMSE': round(rmse, 5), 'R2': round(r2, 4),
            'DA': round(da, 4), 'z': round(z, 3),
            'p_val': round(pval, 4), 'Sig': sig
        })
        print(f"{ticker} | {model_name:<15} RMSE={rmse:.5f} R2={r2:+.4f} "
              f"DA={da:.3f} z={z:+.3f} p={pval:.3f} {'*' if sig else ''}")

# ── Summary table ─────────────────────────────────────────────────────────────
df_res = pd.DataFrame(results)
summary = df_res.groupby('Model').agg(
    Avg_RMSE  = ('RMSE', 'mean'),
    Avg_R2    = ('R2',   'mean'),
    Avg_DA    = ('DA',   'mean'),
    Sig_Count = ('Sig',  'sum'),
).round(4)
summary['Sig_Count'] = summary['Sig_Count'].astype(int)
summary = summary.sort_values('Avg_DA', ascending=False)

print("\n=== SUMMARY TABLE (averaged across 8 tickers) ===")
print(summary.to_string())
print(f"\nTotal significant ticker-model pairs: {df_res['Sig'].sum()} / {len(df_res)}")

# ── Per-ticker DA significance ────────────────────────────────────────────────
print("\n=== ElasticNet DA per ticker ===")
en = df_res[df_res['Model']=='ElasticNet'][['Ticker','DA','z','p_val','Sig']]
print(en.to_string(index=False))


ASBN | ElasticNet      RMSE=0.20866 R2=-0.0119 DA=0.486 z=-0.844 p=0.801 
ASBN | SVR             RMSE=0.20183 R2=+0.0532 DA=0.429 z=-4.352 p=1.000 
ASBN | KNN             RMSE=0.19833 R2=+0.0858 DA=0.494 z=-0.390 p=0.652 
ASBN | RandomForest    RMSE=0.21219 R2=-0.0465 DA=0.522 z=+1.364 p=0.086 
ASBN | XGBoost         RMSE=0.21889 R2=-0.1136 DA=0.502 z=+0.130 p=0.448 
ASBN | LightGBM        RMSE=0.21735 R2=-0.0980 DA=0.480 z=-1.234 p=0.891 
CCBN | ElasticNet      RMSE=0.09200 R2=-0.0117 DA=0.712 z=+13.056 p=0.000 *
CCBN | SVR             RMSE=0.09581 R2=-0.0972 DA=0.622 z=+7.535 p=0.000 *
CCBN | KNN             RMSE=0.09040 R2=+0.0232 DA=0.710 z=+12.926 p=0.000 *
CCBN | RandomForest    RMSE=0.09047 R2=+0.0216 DA=0.723 z=+13.706 p=0.000 *
CCBN | XGBoost         RMSE=0.08979 R2=+0.0364 DA=0.697 z=+12.147 p=0.000 *
CCBN | LightGBM        RMSE=0.09538 R2=-0.0872 DA=0.686 z=+11.432 p=0.000 *
HSBK | ElasticNet      RMSE=0.07468 R2=-0.2603 DA=0.579 z=+4.872 p=0.000 *
HSBK | SVR             RMS

## Reviewer Checks

Quick verification cells for key reviewer concerns (Points 2, 7, 8). Full statistical tests in `000_Statistical_Validation.ipynb`.

In [4]:
# ─── REVIEWER CHECKS ────────────────────────────────────────────────────────
# Added to address Points 2, 3, and 8 from reviewer comments.
# Full statistical validation is in: 000_Statistical_Validation.ipynb

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path(r"C:\Users\asus\Article-10-06-2026-ML-portfoliov1")

# Load metrics
try:
    metrics_unc = pd.read_csv(DATA_PATH / 'portfolio_metrics_unconstrained.csv', index_col=0)
    cum_unc     = pd.read_csv(DATA_PATH / 'cumulative_returns_unconstrained.csv',
                              index_col=0, parse_dates=True)

    # ── CHECK 1 (Reviewer Point 2): CCBN-only vs best strategy ───────────────
    print('='*65)
    print('REVIEWER CHECK 1: Does CCBN-only benchmark beat ML strategies?')
    print('='*65)
    if 'CCBN-only Benchmark' in metrics_unc.index:
        ccbn_sharpe = metrics_unc.loc['CCBN-only Benchmark', 'Sharpe']
        print(f'  CCBN-only Sharpe  : {ccbn_sharpe:.4f}')
        for strat in metrics_unc.index:
            sh = metrics_unc.loc[strat, 'Sharpe']
            beats = 'BEATS CCBN' if sh > ccbn_sharpe else 'below CCBN'
            print(f'  {strat:<35}: Sharpe={sh:.4f}  [{beats}]')
    else:
        print('  CCBN-only not found — run revised portfolio notebook first.')

    # ── CHECK 2 (Reviewer Point 7): Rank-tilt vs raw-ML ablation ─────────────
    print()
    print('='*65)
    print('REVIEWER CHECK 2: Rank-tilt ±5% vs Raw ML-μ ablation')
    print('='*65)
    for s in ['ML-Enhanced MVO (rank±5%)', 'ML-MVO Raw μ (ablation)']:
        if s in metrics_unc.index:
            row = metrics_unc.loc[s]
            print(f'  {s:<40}: Sharpe={row["Sharpe"]:.4f}  Return={row["Ann. Return (%)"]:+.2f}%')
        else:
            print(f'  {s} — not found.')

    # ── CHECK 3 (Reviewer Point 8): BL sensitivity already in portfolio nb ───
    print()
    bl_mu = pd.read_csv(DATA_PATH / 'bl_mu_vectors.csv')
    print('BL mu vectors loaded. See portfolio notebook Section 4.3 for sensitivity.')
    print(bl_mu.to_string(index=False))

except Exception as e:
    print(f'Error: {e}')
    import os
    print('Files in folder:')
    try:
        for ff in os.listdir(DATA_PATH): print(' -', ff)
    except: pass

Error: [Errno 2] No such file or directory: 'C:\\Users\\asus\\Article-10-06-2026-ML-portfoliov1\\portfolio_metrics_unconstrained.csv'
Files in folder:
 - .ipynb_checkpoints
 - 0000-Check-data-article_revised.ipynb
 - 000Final_ml-Articlev3_revised.ipynb
 - 000Final_portfolio_Articlev3_revised.ipynb
 - 000_Statistical_Validation.ipynb
 - view_dl_results_revised.ipynb
